# Module 2 — Vector Store & Retrieval

**Duration:** ~90 minutes  
**Goal:** Index your chunks into ChromaDB and build a retrieval system that finds semantically relevant passages.

---

## How a vector store works

```
           Indexing (offline)                    Querying (online)
  ─────────────────────────────         ──────────────────────────────────
  Document → Chunks → Embeddings        Question → Embedding → ANN Search
                          │                                        │
                     ChromaDB ◀──────────────────────────────────▶ Top-k chunks
```

ChromaDB stores:
- The **document text** (for returning to the LLM)
- The **embedding vector** (for similarity search)
- **Metadata** (source, date, etc.) for filtering

### Similarity metrics

| Metric | Range | Best for |
|--------|-------|----------|
| Cosine similarity | 0–1 | Semantic meaning |
| L2 (Euclidean) | 0–∞ | When magnitude matters |
| Inner product | -∞–∞ | Normalised embeddings |

ChromaDB defaults to **cosine similarity** — correct for sentence-transformer embeddings.

In [ ]:
import json
import re
from pathlib import Path

import chromadb
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

# Load corpus
with open("../data/raw/corpus.json") as f:
    corpus = json.load(f)

# Reuse chunking function from Module 1
def chunk_fixed_size(text: str, chunk_size: int = 200, overlap: int = 20) -> list[str]:
    words = text.split()
    step = chunk_size - overlap
    return [" ".join(words[i : i + chunk_size]) for i in range(0, len(words), step) if words[i:i+chunk_size]]

# Prepare all chunks
all_chunks = []
for article in corpus:
    for i, text in enumerate(chunk_fixed_size(article["content"])):
        all_chunks.append({"text": text, "source": article["title"], "id": f"{article['title']}_{i}"})

print(f"Total chunks: {len(all_chunks)}")

# Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")

---
## Part 1 — Setting up ChromaDB

### Exercise 2.1 — Create a persistent ChromaDB collection

> **Task:** Create a ChromaDB client that persists to disk (so we don't re-embed every time),  
> then create (or get) a collection called `"workshop_rag"`.
>
> **Hints:**
> - `chromadb.PersistentClient(path="...")` stores data on disk
> - `client.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})` creates or opens a collection
> - Check `collection.count()` to see if it's already populated

In [ ]:
CHROMA_PATH = "../chroma_db"

# TODO: create a persistent ChromaDB client
chroma_client = ...

# TODO: get or create collection with cosine similarity
collection = ...

print(f"Collection '{collection.name}' has {collection.count()} documents")

### Exercise 2.2 — Index the chunks

> **Task:** Embed all chunks and add them to the collection.
>
> Skip indexing if the collection is already populated (idempotent).
>
> **Hints:**
> - `collection.add(documents=[...], embeddings=[...], ids=[...], metadatas=[...])`
> - `embeddings` must be a Python list of lists (`.tolist()` on a numpy array)
> - ChromaDB `ids` must be unique strings
> - ChromaDB processes in batches — add up to 5000 at a time to be safe
> - `tqdm` can give you a progress bar

In [ ]:
from tqdm import tqdm
import time

BATCH_SIZE = 500

def index_chunks(collection, chunks: list[dict], embed_model) -> None:
    """Embed and index chunks into ChromaDB. Skips if already populated."""
    if collection.count() >= len(chunks):
        print(f"Collection already has {collection.count()} docs — skipping indexing")
        return

    print(f"Indexing {len(chunks)} chunks...")
    start = time.time()

    # TODO: Deduplicate chunks by id first (the dataset may contain duplicate documents).
    # Then loop over unique chunks in batches of BATCH_SIZE.
    # For each batch:
    #   1. Extract texts, ids, and metadatas
    #   2. Embed the texts
    #   3. Upsert to collection (use upsert instead of add so re-running is safe)
    pass


index_chunks(collection, all_chunks, embed_model)
print(f"Collection now has {collection.count()} documents")

---
## Part 2 — Retrieval

### Exercise 2.3 — Build a retrieval function

> **Task:** Implement `retrieve(question, top_k)` that:
> 1. Embeds the question
> 2. Queries ChromaDB for the top-k most similar chunks
> 3. Returns results with text, source, and similarity score
>
> **Hints:**
> - `collection.query(query_embeddings=[[...]], n_results=top_k)` returns a dict
> - The result contains `documents`, `metadatas`, `distances` (ChromaDB returns distances, not similarities)
> - With cosine space: `similarity = 1 - distance`

In [ ]:
def retrieve(question: str, top_k: int = 5) -> list[dict]:
    """
    Retrieve the top-k most relevant chunks for a question.

    Returns a list of dicts with keys: text, source, similarity
    """
    # TODO: embed the question
    q_embedding = ...

    # TODO: query ChromaDB
    results = ...

    # TODO: package results into a clean list of dicts
    retrieved = []
    # ...
    return retrieved


# Test retrieval
question = "What hospitalization costs does DKV cover?"
results = retrieve(question, top_k=5)

print(f"Query: {question}\n")
for i, r in enumerate(results, 1):
    print(f"[{i}] Source: {r.get('source', '?')} | Similarity: {r.get('similarity', 0):.3f}")
    print(f"    {r.get('text', '')[:150]}...\n")

---
## Part 3 — Retrieval Quality Analysis

### Exercise 2.4 — Experiment with retrieval parameters

> **Task:** Use the code below to explore how changing `top_k` and the similarity threshold affects what you get back.
>
> **Questions to answer:**
> 1. At what similarity score does retrieved text stop being relevant?
> 2. What `top_k` gives enough context without overwhelming the LLM?
> 3. Try a question that spans two topics — do you get results from both?

In [ ]:
# Experiment: visualise similarity scores for different queries
test_questions = [
    "What hospitalization costs does DKV cover?",
    "What dental treatments does DKV Smile cover?",
    "What does DKV Home Care insurance provide?",
    "What is the capital of France?",  # out-of-domain question
]

fig, axes = plt.subplots(1, len(test_questions), figsize=(16, 4))

for ax, question in zip(axes, test_questions):
    results = retrieve(question, top_k=10)
    similarities = [r.get("similarity", 0) for r in results]
    sources = [r.get("source", "")[:20] for r in results]

    bars = ax.barh(range(len(similarities)), similarities, color="steelblue")
    ax.set_yticks(range(len(sources)))
    ax.set_yticklabels(sources, fontsize=7)
    ax.set_xlim(0, 1)
    ax.axvline(0.4, color="red", linestyle="--", alpha=0.5, label="threshold")
    ax.set_title(question[:40] + "...", fontsize=8, wrap=True)
    ax.set_xlabel("Similarity")

plt.tight_layout()
plt.show()

print("\nNote the out-of-domain question — what's the highest similarity score?")
print("This is the 'retrieval noise floor' — you may want a minimum similarity threshold.")

In [ ]:
# Exercise 2.4 extension: add a similarity threshold filter
def retrieve_filtered(question: str, top_k: int = 5, min_similarity: float = 0.3) -> list[dict]:
    """Retrieve chunks above a minimum similarity threshold."""
    results = retrieve(question, top_k=top_k)
    # TODO: filter out results below min_similarity
    return results


# Try it with the out-of-domain question
ood_results = retrieve_filtered("What is the capital of France?", top_k=5, min_similarity=0.3)
print(f"Filtered results for OOD question: {len(ood_results)} (should be fewer or 0)")

---
## Part 4 — Metadata Filtering

ChromaDB supports filtering results by metadata — very useful for multi-domain corpora.

### Exercise 2.5 — Source-filtered retrieval

> **Task:** Modify `retrieve` to accept an optional `source_filter` parameter.  
> When provided, only return chunks from that source article.
>
> **Hint:** `collection.query(..., where={"source": "Insurance"})` filters by metadata

In [ ]:
def retrieve_from_source(question: str, source: str, top_k: int = 5) -> list[dict]:
    """Retrieve chunks only from a specific source article."""
    # TODO: add where filter to ChromaDB query
    pass


# Test: retrieve only from the Reinsurance article
results = retrieve_from_source("What accommodation costs are covered during hospitalisation?", source="dkv_hospitalisation_en", top_k=3)
print(f"Results from 'dkv_hospitalisation_en' only: {len(results)}")
for r in results:
    print(f"  Source: {r.get('source')} | {r.get('text', '')[:100]}...")

---
## Part 5 — Chunking Impact on Retrieval Quality

### Exercise 2.6 — Does chunk size affect retrieval?

In notebook 02 you experimented with different chunk sizes.
Here you will index the same corpus with two configurations and compare
which retrieval results are more relevant for the same question.

This closes the loop: chunk size is not just a structural parameter —
it directly changes what the retrieval system finds.

> **Task:** Run the cell below as-is, then try changing `CONFIGS` to test your own settings from Exercise 1.4b.

In [ ]:
# Compare retrieval quality across two chunk configurations.
# Uses in-memory ChromaDB collections — no writes to chroma_db/.
from langchain_text_splitters import RecursiveCharacterTextSplitter as RCS

CONFIGS = {
    "small_400":   {"chunk_size": 400,  "chunk_overlap": 50},
    "default_800": {"chunk_size": 800,  "chunk_overlap": 100},
}

TEST_QUESTION = "What hospitalization costs does DKV cover?"

for config_name, params in CONFIGS.items():
    splitter = RCS(separators=["\n\n", "\n", ". ", " ", ""], **params)
    exp_chunks = []
    for doc in corpus:
        for i, text in enumerate(splitter.split_text(doc["content"])):
            exp_chunks.append({"text": text, "source": doc["title"], "id": f"{doc['title']}_{i}"})

    exp_client = chromadb.EphemeralClient()
    exp_col = exp_client.get_or_create_collection(
        name=config_name, metadata={"hnsw:space": "cosine"}
    )
    texts = [c["text"] for c in exp_chunks]
    ids   = [c["id"]   for c in exp_chunks]
    metas = [{"source": c["source"]} for c in exp_chunks]
    embs  = embed_model.encode(texts, batch_size=64, show_progress_bar=False).tolist()
    exp_col.add(documents=texts, embeddings=embs, ids=ids, metadatas=metas)

    q_emb = embed_model.encode(TEST_QUESTION).tolist()
    res = exp_col.query(
        query_embeddings=[q_emb], n_results=3,
        include=["documents", "distances"]
    )

    print(f"\n\u2500\u2500 {config_name} ({len(exp_chunks)} chunks) \u2500\u2500")
    for rank, (text, dist) in enumerate(zip(res["documents"][0], res["distances"][0]), 1):
        sim = round(1.0 - dist, 3)
        print(f"  [{rank}] sim={sim:.3f}  {text[:120].replace(chr(10), ' ')}...")

print(f"\nQuestion: {TEST_QUESTION}")
print("Discussion: Which config returns more focused, relevant passages? Why?")

---
## Reflection Questions

1. **Threshold tuning:** What happens if your similarity threshold is too high? Too low?

2. **ANN vs exact search:** ChromaDB uses Approximate Nearest Neighbour (ANN/HNSW) search.  
   It's fast but not always exact. When would exactness matter?

3. **Index freshness:** How would you update the index when documents change?
   Hint: ChromaDB has `collection.update()` and `collection.delete()`.

4. **Multi-vector retrieval:** What if a question is complex and needs context from multiple topics?

---

## What we built

- [x] Persistent ChromaDB collection
- [x] Batch indexing with metadata
- [x] Semantic retrieval with similarity scores
- [x] Threshold filtering
- [x] Metadata-based source filtering

## Next: Module 3 — Full RAG Pipeline

We have retrieval working. Now we'll connect it to Claude and build the complete pipeline.  
Open `04_rag_pipeline.ipynb` →